In [ ]:
!pip install -q streamlit pandas numpy plotly scikit-learn
!wget -q -O - https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 > /tmp/cloudflared
!chmod +x /tmp/cloudflared

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 42.8 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import io
import plotly.graph_objects as go
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier

# --- 1. HYPER-LOCAL DATASET WITH GPS ---
csv_data = """Career_Name,Domain,Base_Cost_INR,Starting_Salary,Institution_Tier,STEAM_Ecosystem,lat,lon
AI & Machine Learning,Technology,1200000,1200000,Tier 1,IIT Madras AI Hub,12.9915,80.2337
Data Engineering,Technology,800000,850000,Tier 2,TIDEL Park Trichy,10.7905,78.7047
EV Battery Engineering,Engineering,1500000,900000,Tier 1,Hosur EV Cluster,12.7409,77.8253
Agri-Tech Robotics,Engineering,450000,600000,Tier 2,TNAU Coimbatore,11.0168,76.9558
UI/UX Architecture,Design,600000,650000,Tier 2,Chennai Startup Incubators,13.0827,80.2707
Game & Animation Design,Arts,800000,450000,Tier 3,Chennai Gaming Studios,12.9815,80.2237
Biomedical Research,Science,1200000,700000,Tier 1,TICEL Bio Park Chennai,12.9791,80.2459
FinTech Analytics,Commerce,900000,800000,Tier 2,Chennai Financial City,13.0500,80.2800"""

df = pd.read_csv(io.StringIO(csv_data))

# --- 2. ADVANCED ML & FINTECH LOGIC ---
@st.cache_resource
def train_success_model():
    np.random.seed(42)
    X_train = np.random.randint(0, 4, size=(1000, 6))
    y_train = []
    for row in X_train:
        score = (row[0] * 1.2) + (row[1] * 1.5) + (row[2] * 1.0) - (row[3] * 1.2) - (row[4] * 1.5) - (row[5] * 0.5)
        y_train.append(1 if score > 1.5 else 0)
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    return model

ml_model = train_success_model()

def calculate_emi(principal, rate=0.09, tenure_years=7):
    if principal <= 0: return 0
    r = rate / 12
    n = tenure_years * 12
    emi = principal * r * ((1 + r)**n) / (((1 + r)**n) - 1)
    return round(emi)

def get_institution_loan_cap(tier):
    # Base caps on actual Indian banking collateral-free limits (SBI Scholar / PM Vidyalaxmi)
    return 5000000 if tier == "Tier 1" else (1500000 if tier == "Tier 2" else 750000)

def create_gauge(prob):
    color = "#00FFAA" if prob > 75 else "#FFD700" if prob > 40 else "#FF4444"
    fig = go.Figure(go.Indicator(
        mode="gauge+number", value=prob, title={'text': "ML Graduation Probability", 'font': {'size': 16}},
        gauge={'axis': {'range': [0, 100]}, 'bar': {'color': color}, 'bgcolor': "rgba(0,0,0,0)"}
    ))
    fig.update_layout(height=250, margin=dict(t=30, b=10, l=10, r=10), paper_bgcolor="rgba(0,0,0,0)")
    return fig

def create_radar(stem, grit, fin_stress, capital, learn_val, stress_val, roi_val):
    # Dynamic values replacing hardcoded ones
    categories = ['Academic Readiness', 'Grit/Resilience', 'Financial Power', 'Risk Tolerance', 'Market Demand (ROI Goal)']
    s_vector = [stem * 3.3, grit * 3.3, 5, max(1, 10 - (stress_val * 3.3)), (learn_val * 5)]
    p_vector = [7, 6, min(10, capital / 150000), max(0, 10 - (fin_stress * 3.3)), max(2, 10 - (roi_val * 3.3))]
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=s_vector, theta=categories, fill='toself', name='Student Vector', line_color='#00F0FF'))
    fig.add_trace(go.Scatterpolar(r=p_vector, theta=categories, fill='toself', name='Parent Vector', line_color='#FF9900'))
    fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 10]), bgcolor="rgba(0,0,0,0)"),
                      showlegend=True, height=280, margin=dict(t=20, b=20, l=20, r=20), paper_bgcolor="rgba(0,0,0,0)")
    return fig

# --- 3. UI DASHBOARD ---
st.set_page_config(page_title="PRISM 11.0: FinTech Engine", layout="wide")
st.title("💠 PRISM Engine 11.0: Advanced FinTech & ML Routing")

tab1, tab2, tab3 = st.tabs(["🎓 Student Profiling", "💼 Parent Financials", "🚀 Market & EMI Synthesis"])

with tab1:
    col1, col2 = st.columns(2)
    with col1:
        stem_band = st.selectbox("1. 12th Board score or TNEA cutoff?", ["Top Tier (>190 / >90%)", "Strong (170–189 / 80–89%)", "Average (150–169 / 70–79%)", "Below 150 (<70%)"])
        learn_style = st.selectbox("2. How do you understand concepts fastest?", ["Working through math & code", "Hands-on tinkering", "Visuals & diagrams", "Reading guides"])
        self_learn = st.selectbox("3. Self-Directed Learning (Weekly)?", ["10+ hours", "5–10 hours", "Under 5 hours"])
        stress_resp = st.selectbox("4. Failure Response?", ["Take a breath & debug", "Keep grinding", "Feel stuck"])
    with col2:
        grit = st.selectbox("5. Extracurricular Portfolio?", ["Active GitHub / live creations", "Small school projects", "None yet"])
        primary_motive = st.selectbox("6. Primary Motivation?", ["Financial independence", "Technological innovation", "Creative expression", "Social impact"])
        student_domain = st.selectbox("7. Preferred Domain?", ["Technology", "Engineering", "Design", "Science", "Arts", "Commerce"])
        hostel_req = st.radio("8. Geographic Flexibility?", ["Commute daily (Day Scholar)", "Relocate to a hostel"])

    stem_val = {"Top Tier (>190 / >90%)": 3, "Strong (170–189 / 80–89%)": 2, "Average (150–169 / 70–79%)": 1, "Below 150 (<70%)": 0}[stem_band]
    grit_val = {"Active GitHub / live creations": 3, "Small school projects": 1, "None yet": 0}[grit]
    learn_val = {"10+ hours": 2, "5–10 hours": 1, "Under 5 hours": 0}[self_learn]
    stress_val = {"Take a breath & debug": 0, "Keep grinding": 1, "Feel stuck": 2}[stress_resp]

with tab2:
    col3, col4 = st.columns(2)
    with col3:
        income_bracket = st.selectbox("1. Yearly family income?", ["Under ₹2.5 Lakhs (Fee-waiver)", "₹2.5L – ₹8 Lakhs", "Above ₹8 Lakhs"])
        savings = st.number_input("2. Cash savings for college upfront?", min_value=0, max_value=5000000, value=300000, step=50000)
        gold_asset = st.selectbox("3. Backup assets (gold, land)?", ["Yes (~₹5L buffer)", "No (Cash/loan only)"])
        quota = st.checkbox("4. First-Generation Graduate Eligible")
    with col4:
        dependents = st.number_input("5. Dependent children?", min_value=1, max_value=6, value=2)
        dti_status = st.selectbox("6. Parent's Current Loan EMI burden?", ["Minimal (<20%)", "Moderate (20%–40%)", "Heavy (>40%)"])
        roi_horizon = st.selectbox("7. Expected ROI Horizon?", ["Right after college (6–12 mos)", "2 to 3 years", "Open to 4+ years"])
        parent_veto = st.multiselect("8. Domain Veto (Will NOT fund)?", ["Technology", "Engineering", "Design", "Science", "Arts", "Commerce"])

    fin_stress_val = {"Minimal (<20%)": 0, "Moderate (20%–40%)": 1, "Heavy (>40%)": 3}[dti_status]
    roi_val = {"Right after college (6–12 mos)": 0, "2 to 3 years": 1, "Open to 4+ years": 2}[roi_horizon]
    asset_buffer = 500000 if "Yes" in gold_asset else 0
    # True capital caps dependent limit; bounded to 1-3 for ML normalization
    true_capital = (savings + asset_buffer) / dependents
    dep_normalized = min(3, dependents)

with tab3:
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Adjusted Parent Seed Capital", f"₹{int(true_capital):,}", "Per Dependent")
    m2.metric("State Concession Status", "Active ✅" if "Under ₹2.5" in income_bracket or quota else "Not Eligible ❌")
    m3.metric("Geographic Cost Modifier", "+ ₹4.0L" if "hostel" in hostel_req else "₹0 (Local)")
    m4.metric("Parent Debt Health", "Stretched" if fin_stress_val == 3 else "Stable")
    st.markdown("---")

    # Engine Processing with FinTech Student DTI Logic
    results = []
    top_loan_needed = 0

    for index, row in df.iterrows():
        if row["Domain"] in parent_veto: continue

        cost = row["Base_Cost_INR"]
        if "hostel" in hostel_req: cost += 400000
        if "Under ₹2.5" in income_bracket or quota: cost *= 0.6

        required_loan = max(0, cost - true_capital)
        institution_cap = get_institution_loan_cap(row["Institution_Tier"])

        student_monthly_salary = row["Starting_Salary"] / 12
        student_emi = calculate_emi(required_loan)
        student_dti_ratio = (student_emi / student_monthly_salary) * 100 if student_monthly_salary > 0 else 100

        # New Logic Gate Based on Mentor Feedback
        if required_loan > institution_cap:
            affordability = f"❌ Blocked: Exceeds Tier Loan Cap (Max ₹{int(institution_cap/100000)}L)"
        elif student_dti_ratio > 35:
            affordability = f"❌ Blocked: Future Debt Trap (EMI >35% of Salary)"
        elif required_loan > 1000000 and "Right after college" in roi_horizon:
            affordability = "❌ Blocked: Parent ROI Horizon Mismatch"
        else:
            affordability = "Funded (No Loan)" if required_loan == 0 else f"✅ EMI: ₹{student_emi:,}/mo"
            results.append({
                "Career": row["Career_Name"],
                "Domain": row["Domain"],
                "Adjusted_Cost": cost,
                "Projected_Salary": row["Starting_Salary"],
                "Student_EMI": student_emi,
                "ROI_Score": round(row["Starting_Salary"] / cost, 2),
                "Institution": row["Institution_Tier"],
                "Hub": row["STEAM_Ecosystem"],
                "lat": row["lat"],
                "lon": row["lon"],
                "Loan_Needed": required_loan
            })
            if required_loan > top_loan_needed: top_loan_needed = required_loan

    results_df = pd.DataFrame(results)

    v1, v2, v3 = st.columns([1, 1, 1])
    with v1:
        st.subheader("ML Dropout Prediction")
        student_features = np.array([[stem_val, grit_val, learn_val, stress_val, fin_stress_val, dep_normalized]])
        success_prob = ml_model.predict_proba(student_features)[0][1] * 100
        st.plotly_chart(create_gauge(success_prob), use_container_width=True)
    with v2:
        st.subheader("Parent-Student Vector")
        st.plotly_chart(create_radar(stem_val, grit_val, fin_stress_val, true_capital, learn_val, stress_val, roi_val), use_container_width=True)
    with v3:
        st.subheader("Max Capital Allocation")
        if not results_df.empty and top_loan_needed > 0:
            pie_fig = px.pie(values=[true_capital, top_loan_needed], names=['Parent Seed', 'Student Loan'], hole=0.7, color_discrete_sequence=['#00CC96', '#EF553B'])
            pie_fig.update_layout(height=280, margin=dict(t=20, b=20, l=20, r=20), paper_bgcolor="rgba(0,0,0,0)", showlegend=False)
            pie_fig.add_annotation(text=f"Max Loan:<br>₹{int(top_loan_needed/100000)}L", x=0.5, y=0.5, font_size=16, showarrow=False)
            st.plotly_chart(pie_fig, use_container_width=True)
        elif not results_df.empty:
            st.success("✅ 100% Cash/Asset Funded. Zero EMI Required.")
        else:
            st.error("No viable paths.")

    st.markdown("---")

    if not results_df.empty:
        c1, c2 = st.columns([1.2, 1])
        with c1:
            st.subheader("ROI Sweet Spot (Salary vs Cost)")
            bubble_fig = px.scatter(results_df, x="Adjusted_Cost", y="Projected_Salary", size="ROI_Score", color="Institution", hover_name="Career", size_max=40, color_discrete_sequence=px.colors.qualitative.Pastel)
            bubble_fig.update_layout(height=400, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", xaxis_title="Adjusted Cost (INR)", yaxis_title="Year 1 Salary (INR)")
            st.plotly_chart(bubble_fig, use_container_width=True)
        with c2:
            st.subheader("Hyper-Local STEAM Map")
            # Bug fix: use open-street-map for bulletproof tile loading
            map_fig = px.scatter_mapbox(results_df, lat="lat", lon="lon", hover_name="Hub", hover_data=["Career", "Adjusted_Cost"], color="Institution", zoom=5.5, center={"lat": 11.5, "lon": 79.0}, height=400)
            map_fig.update_layout(mapbox_style="open-street-map", margin={"r":0,"t":0,"l":0,"b":0})
            st.plotly_chart(map_fig, use_container_width=True)

        display_df = results_df[["Career", "Institution", "Hub", "Adjusted_Cost", "Projected_Salary", "Student_EMI"]].copy()
        display_df["Adjusted_Cost"] = display_df["Adjusted_Cost"].apply(lambda x: f"₹{int(x):,}")
        display_df["Projected_Salary"] = display_df["Projected_Salary"].apply(lambda x: f"₹{int(x):,}")
        display_df["Student_EMI"] = display_df["Student_EMI"].apply(lambda x: f"₹{int(x):,}")

        st.dataframe(display_df, use_container_width=True)
        st.download_button("📥 Download PRISM Roadmap (CSV)", data=display_df.to_csv(index=False).encode('utf-8'), file_name='prism_roadmap.csv', mime='text/csv', type="primary")
    else:
        st.error("Zero viable pathways found. The Student Debt constraints and Domain Vetoes eliminated all options.")

        # Bug fix: Smart Compromise that dynamically checks the veto list
        allowed_domains = [d for d in ["Technology", "Engineering", "Design", "Science", "Arts", "Commerce"] if d not in parent_veto]

        if allowed_domains:
            suggested_bridge = allowed_domains[0]
            st.warning(f"⚡ **SMART COMPROMISE PROTOCOL ACTIVATED:** Since {', '.join(parent_veto)} is vetoed, PRISM recommends pivoting to a **{suggested_bridge}** pathway (e.g., Regional Diploma or Startup Apprenticeship) that requires zero institutional loans and bypasses the DTI Trap.")
        else:
            st.warning("⚡ **CRITICAL ALERT:** All domains vetoed by parent. Intervention required.")

Writing app.py


In [ ]:
!pkill -f streamlit
import subprocess
import time
subprocess.Popen(['streamlit', 'run', 'app.py'])
time.sleep(5)
!/tmp/cloudflared tunnel --url http://localhost:8501

2026-10-07T20:16:37Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-07T20:16:37Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-07T20:16:43Z INF +--------------------------------------------------------------------------------------------+
2026-10-07T20:16:43Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-07T20:16:43Z INF |  https://capable-issues-surfaces-aviation.trycloudflar